# Imports

In [ ]:
import math

import adjustText
import geopandas as gpd
import matplotlib
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import sklearn
import torch
import zarr
from tqdm import tqdm

from forest_browning.config import (
    FOREST_MASK,
    PX,
    REF_BBOX,
    REF_TRANSFORM,
    SPATIAL_DATASET_ZARR,
    TEMPORAL_DATASET_ZARR,
)
from forest_browning.train import double_logistic_function

In [ ]:
ds_temporal = zarr.open_group(TEMPORAL_DATASET_ZARR, mode="r")
ds_spatial = zarr.open_group(SPATIAL_DATASET_ZARR, mode="r")
params = ds_temporal["params_2"]
params_lower = params["params_lower"]
params_median = params["params_median"]
params_upper = params["params_upper"]
ndvi = ds_temporal["ndvi"]
ndsi = ds_temporal["ndsi"]
anomaly_values = ds_temporal["anomalies"]
anomaly_scores = ds_temporal["anomaly_scores"]

dates = pd.to_datetime([d.decode("utf-8") for d in ds_temporal["dates"][:]])
dtindex = pd.DatetimeIndex(dates)
doy = dtindex.dayofyear.to_numpy()
is_leap = dtindex.is_leap_year.astype(int)
t = torch.tensor((doy - 1) / (365 + is_leap), dtype=torch.float32)

In [ ]:
forest_mask = np.load(FOREST_MASK)
height, width = forest_mask.shape
forest_flat_indices = np.flatnonzero(forest_mask == 1)

# Figure 1

In [ ]:
indices = [10803621, 71181649, 96658205, 43074504]

order = np.argsort(dates)
dates_sorted = np.array(dates)[order]
t_sorted = t[order]

fig, axs = plt.subplots(4, 1, figsize=(8, 8), sharex=True, sharey=True, dpi=600)
axs = axs.flatten()

vmin = np.nanmin([np.nanmin(anomaly_scores[i]) for i in indices])
vmax = -1.5
norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
cmap = plt.cm.magma_r

for i, idx in enumerate(indices):
    lower = (
        double_logistic_function(t_sorted, torch.as_tensor(params_lower[[idx]]).float())
        .squeeze()
        .numpy()
    )
    upper = (
        double_logistic_function(t_sorted, torch.as_tensor(params_upper[[idx]]).float())
        .squeeze()
        .numpy()
    )
    ndvi_vals = np.asarray(ndvi[idx])[order]
    ndsi_vals = np.asarray(ndsi[idx])[order]

    is_valid = (
        (ndvi_vals != -32768)
        & (ndvi_vals != 32767)
        & (ndsi_vals < 4300)
        & ~(np.isnan(ndvi_vals))
        & (ndvi_vals <= 10000)
        & (ndvi_vals > -1000)
    )
    ndvi_valid = ndvi_vals[is_valid] / 10000.0
    dates_valid = dates_sorted[is_valid]

    iqr = np.abs(upper - lower)
    low_thr = lower - 1.5 * iqr
    high_thr = upper + 1.5 * iqr

    ax = axs[i]
    ax.plot(dates_sorted, high_thr, ls="--", lw=1, color="tab:green", alpha=0.8)
    ax.plot(dates_sorted, low_thr, ls="--", lw=1, color="tab:red", alpha=0.8)
    ax.plot(dates_sorted, lower, color="tab:red", lw=1.5)
    ax.plot(dates_sorted, upper, color="tab:green", lw=1.5)
    ax.fill_between(dates_sorted, lower, upper, color="tab:red", alpha=0.1)

    ndvi_interp_low = np.interp(
        dates_valid.astype("datetime64[D]").astype(float),
        dates_sorted.astype("datetime64[D]").astype(float),
        low_thr,
    )
    ndvi_interp_high = np.interp(
        dates_valid.astype("datetime64[D]").astype(float),
        dates_sorted.astype("datetime64[D]").astype(float),
        high_thr,
    )
    is_anomaly = ndvi_valid < ndvi_interp_low

    scores_valid = anomaly_scores[idx][order][is_valid]
    scores_anom = scores_valid[is_anomaly]
    colors = cmap(norm(scores_anom))

    ax.scatter(
        dates_valid[~is_anomaly],
        ndvi_valid[~is_anomaly],
        color="black",
        s=10,
        zorder=3,
        label="Normal",
    )
    ax.scatter(
        dates_valid[is_anomaly],
        ndvi_valid[is_anomaly],
        c=colors,
        s=15,
        zorder=4,
        label="Anomaly",
    )

    ax.set_ylim(-0.1, 1.001)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_ylabel("NDVI")

sm = matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap)
fig.colorbar(
    sm, ax=axs, orientation="vertical", fraction=0.03, pad=0.02, label="Anomaly score"
)

plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.subplots_adjust(hspace=0.05)

plt.savefig("../figs/figure_1.png", bbox_inches="tight", pad_inches=0)

plt.show()

# Figure 2

In [ ]:
date_strings = ["2023-08-22", "2023-08-24"]

scores = ds_spatial["anomaly_scores"]

date_idx_list = []
for ds in date_strings:
    dt = pd.to_datetime(ds)
    matches = np.where(dates == dt)[0]
    date_idx_list.append(int(matches[0]))

selected_scores = []
for idx in date_idx_list:
    col = scores[idx, :].astype(float)
    col = col.astype(float)
    col[col > -1.5] = np.nan
    selected_scores.append(col)

stacked = np.vstack(selected_scores)
merged_scores = np.nanmean(stacked, axis=0)

merged_scores[np.isnan(merged_scores)] = 0

score_raster = np.full((height * width,), np.nan, dtype=float)
score_raster[forest_flat_indices] = merged_scores
score_raster = score_raster.reshape((height, width))

mask = ~np.isnan(score_raster)
valid_rows = np.where(mask.any(axis=1))[0]
valid_cols = np.where(mask.any(axis=0))[0]
score_raster_cropped = score_raster[
    valid_rows[0] : valid_rows[-1] + 1, valid_cols[0] : valid_cols[-1] + 1
]

relative_coor_1 = (19865, 659)
relative_coor_2 = (16865, 10459)
relative_coor_3 = (22966, 20451)

colors_list = plt.cm.Reds(np.linspace(0, 1, 256))
colors_list[-1, :] = [0, 0, 0, 1]
cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
    "Reds_black_cut", colors_list
)

vmin = -7.0
vmax = -1.5
norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)

fig = plt.figure(figsize=(10, 6), dpi=1200)

zoom_coords = [
    (
        relative_coor_1[1],
        relative_coor_1[1] + 100,
        relative_coor_1[0],
        relative_coor_1[0] + 100,
    ),
    (
        relative_coor_2[1],
        relative_coor_2[1] + 100,
        relative_coor_2[0],
        relative_coor_2[0] + 100,
    ),
    (
        relative_coor_3[1],
        relative_coor_3[1] + 100,
        relative_coor_3[0],
        relative_coor_3[0] + 100,
    ),
]

ax_zoom_list = []
for i, (y0, y1, x0, x1) in enumerate(zoom_coords):
    ax_zoom = plt.subplot2grid((3, 11), (i, 0), colspan=2, rowspan=1)
    box = score_raster_cropped[y0:y1, x0:x1]
    ax_zoom.imshow(box, cmap=cmap, norm=norm)
    ax_zoom.axis("off")
    ax_zoom_list.append(ax_zoom)

ax_map = plt.subplot2grid((3, 11), (0, 2), rowspan=3, colspan=9)
cax = ax_map.imshow(score_raster_cropped, cmap=cmap, norm=norm)
ax_map.axis("off")

for y0, y1, x0, x1 in zoom_coords:
    x_min, x_max = min(x0, x1), max(x0, x1)
    y_min, y_max = min(y0, y1), max(y0, y1)
    rect = matplotlib.patches.Rectangle(
        (x_min, y_min),
        x_max - x_min,
        y_max - y_min,
        linewidth=1.2,
        edgecolor="cyan",
        facecolor="none",
        linestyle="--",
    )
    ax_map.add_patch(rect)

cb = fig.colorbar(cax, ax=ax_map, fraction=0.046, pad=0.02)
cb.ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig("../figs/figure_2.png", bbox_inches="tight", pad_inches=0)
plt.show()

# Figure 3

In [ ]:
dfs = []
labels = ["Lower (q=0.25)", "Median (q=0.5)", "Upper (q=0.75)"]
colors = ["tab:red", "tab:blue", "tab:green"]

for csv, label, color in zip(
    [
        "../data/d2_per_doy_lower.csv",
        "../data/d2_per_doy_median.csv",
        "../data/d2_per_doy_upper.csv",
    ],
    labels,
    colors,
):
    df = pd.read_csv(csv).sort_values("doy")
    df["d2_smooth"] = df["d2_pinball"].rolling(7, center=True, min_periods=1).mean()
    dfs.append((df, label, color))

fig, ax = plt.subplots(figsize=(9, 5), dpi=600)
for df, label, color in dfs:
    ax.plot(df["doy"], df["d2_pinball"], lw=1, alpha=0.2, color=color)
    ax.plot(df["doy"], df["d2_smooth"], lw=2.2, color=color, label=label)

ax.set_xlabel("Day of year", fontsize=13)
ax.set_ylabel("$D^2_{\\mathrm{pinball}}$", fontsize=13)
ax.set_title(
    "Quantile regression skill per day of year (relative to climatology baseline)",
    fontsize=14,
    pad=10,
)
ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)
ax.set_ylim(-0.5, 0.5)
ax.legend(loc="lower right", fontsize=12)
plt.tight_layout()
plt.savefig("../figs/figure_3.png", bbox_inches="tight", pad_inches=0)
plt.show()

# Figure 4

In [ ]:
df = pd.read_csv("../data/ndvi_anomaly_stats.csv").sort_values("doy")
df["frac_neg"] = df["neg_anom_count"] / df["total_valid"]

# Apply rolling smoothing
df["frac_neg_smooth"] = df["frac_neg"].rolling(7, center=True, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(9, 5), dpi=600)
ax.plot(df["doy"], df["frac_neg"], lw=1, alpha=0.2, color="tab:red")
ax.plot(
    df["doy"],
    df["frac_neg_smooth"],
    lw=2.2,
    color="tab:red",
    label="Negative anomalies",
)

# Add dashed vertical lines and season labels
season_bounds = [0, 59, 151, 243, 334, 365]
season_labels = ["Winter", "Spring", "Summer", "Autumn"]

y_top = max(df[["frac_neg"]].max().max() * 1.1, 0.05)
ax.set_ylim(0, y_top)

for i, (start, end) in enumerate(zip(season_bounds[:-2], season_bounds[1:-1])):
    ax.axvline(end, color="gray", linestyle="--", lw=0.8, alpha=0.5)
    ax.text(
        start + 5,
        y_top * 0.97,
        season_labels[i],
        ha="left",
        va="top",
        fontsize=11,
        color="dimgray",
    )

# Show overall average fractions as horizontal lines
overall_frac_neg = df["neg_anom_count"].sum() / df["total_valid"].sum()
print("Overall negative anomaly fraction:", overall_frac_neg)
ax.axhline(
    overall_frac_neg,
    color="tab:red",
    ls=":",
    lw=1.5,
    label="Overall negative anomaly fraction",
)

ax.set_xlim(0, 365)
ax.set_xticks([0, 60, 120, 180, 240, 300, 360])
ax.set_xlabel("Day of year", fontsize=13)
ax.set_ylabel("Fraction of pixels", fontsize=13)
ax.set_title(
    "Seasonal frequency of NDVI anomalies\n(fraction of forest pixels per DOY)",
    fontsize=14,
    pad=10,
)
ax.legend(loc="upper right", bbox_to_anchor=(1, 0.94))

ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)

ax.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig("../figs/figure_4.png", bbox_inches="tight")
plt.show()

# Figure 5

In [ ]:
anom = ds_temporal["anomalies"]

n_pixels, n_time = anom.shape
chunk_size = 4000
neg_frac = np.empty(n_pixels, dtype=np.float32)

for i in tqdm(range(0, n_pixels, chunk_size)):
    slc = slice(i, min(i + chunk_size, n_pixels))
    block = anom[slc, :]  # shape (chunk_size, n_time)
    valid = (block != 127) & (block != -128)
    neg = block == -1
    neg_frac[slc] = np.sum(neg & valid, axis=1) / np.sum(valid, axis=1)

In [ ]:
fraction_raster = np.full((height * width,), np.nan, dtype=float)
fraction_raster[forest_flat_indices] = neg_frac
fraction_raster = fraction_raster.reshape((height, width))

mask = ~np.isnan(fraction_raster)

valid_rows = np.where(mask.any(axis=1))[0]
valid_cols = np.where(mask.any(axis=0))[0]

r0, r1 = valid_rows[0], valid_rows[-1]
c0, c1 = valid_cols[0], valid_cols[-1]

fraction_raster_cropped = fraction_raster[r0 : r1 + 1, c0 : c1 + 1]

In [ ]:
sample_coor = (2689600, 1154533)
relative_coor_1 = (
    int((sample_coor[0] - REF_BBOX.left) / 10),
    int((REF_BBOX.top - sample_coor[1]) / 10),
)

sample_coor = (2550435, 1200702)
relative_coor_2 = (
    int((sample_coor[0] - REF_BBOX.left) / 10),
    int((REF_BBOX.top - sample_coor[1]) / 10),
)

relative_coor_3 = (21569, 19302)

In [ ]:
row_offset, col_offset = r0, c0

zoom_coords = [
    (
        relative_coor_2[1] - 50 - row_offset,
        relative_coor_2[1] + 50 - row_offset,
        relative_coor_2[0] - 50 - col_offset,
        relative_coor_2[0] + 50 - col_offset,
    ),
    (
        relative_coor_1[1] - 50 - row_offset,
        relative_coor_1[1] + 50 - row_offset,
        relative_coor_1[0] - 50 - col_offset,
        relative_coor_1[0] + 50 - col_offset,
    ),
    (
        relative_coor_3[1] - 50 - row_offset,
        relative_coor_3[1] + 50 - row_offset,
        relative_coor_3[0] - 50 - col_offset,
        relative_coor_3[0] + 50 - col_offset,
    ),
]

In [ ]:
vmin, vmax = 0.005, 0.1
norm = matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax)

fig = plt.figure(figsize=(10, 6), dpi=1200)

ax_zoom_list = []
for i, (y0, y1, x0, x1) in enumerate(zoom_coords):
    ax_zoom = plt.subplot2grid((3, 9), (i, 0), colspan=2, rowspan=1)
    box = fraction_raster_cropped[y0:y1, x0:x1]
    ax_zoom.imshow(box, cmap="magma", norm=norm)
    ax_zoom.axis("off")
    ax_zoom_list.append(ax_zoom)

ax_map = plt.subplot2grid((3, 9), (0, 2), rowspan=3, colspan=6)
cax = ax_map.imshow(fraction_raster_cropped, cmap="magma", norm=norm)
ax_map.axis("off")

for y0, y1, x0, x1 in zoom_coords:
    x_min, x_max = min(x0, x1), max(x0, x1)
    y_min, y_max = min(y0, y1), max(y0, y1)
    rect = matplotlib.patches.Rectangle(
        (x_min, y_min),
        x_max - x_min,
        y_max - y_min,
        linewidth=1.2,
        edgecolor="cyan",
        facecolor="none",
        linestyle="--",
    )
    ax_map.add_patch(rect)

cb = fig.colorbar(cax, ax=ax_map, fraction=0.046, pad=0.02)
cb.set_ticks([vmin, 0.01, 0.02, 0.05, vmax])
cb.ax.tick_params(labelsize=9)
cb.ax.text(
    0.5,
    -0.03,
    f"< {vmin}",
    ha="center",
    va="top",
    fontsize=8,
    transform=cb.ax.transAxes,
)
cb.ax.text(
    0.5,
    1.03,
    f"> {vmax}",
    ha="center",
    va="bottom",
    fontsize=8,
    transform=cb.ax.transAxes,
)

vals = fraction_raster_cropped.flatten()
vals = vals[~np.isnan(vals)]
ax_hist = plt.subplot2grid((3, 9), (0, 8), rowspan=3, colspan=1)
bins = np.logspace(np.log10(vmin), np.log10(vmax), 50)
ax_hist.hist(
    vals, bins=bins, orientation="horizontal", color="lightgray", edgecolor="black"
)
ax_hist.set_yscale("log")
ax_hist.set_ylim(vmin, vmax)
ax_hist.tick_params(axis="y", left=False, right=False, labelleft=False)

plt.tight_layout()
plt.savefig("../figs/figure_5.png", dpi=300)
plt.show()

# Figure 6

In [ ]:
def get_pixels_from_polygon(polygon, forest_mask, left, top, px, width, height):
    """Get the indices of masked pixels that fall within a given polygon, based on the forest mask and spatial parameters.

    Args:
        polygon (shapely.geometry.Polygon): The polygon for which to find the masked pixels.
        forest_mask (numpy.ndarray): A boolean array indicating the forest mask.
        left (float): The left edge of the image.
        top (float): The top edge of the image.
        px (float): The pixel size.
        width (int): The width of the image.
        height (int): The height of the image.

    Raises:
        RuntimeError: If no masked pixels are found within the polygon.

    Returns:
        list: A list of indices of the masked pixels that fall within the polygon.
    """
    x_min, y_min, x_max, y_max = polygon.bounds

    col_min = max(0, min(width - 1, int(math.floor((x_min - left) / px))))
    col_max = max(0, min(width - 1, int(math.floor((x_max - left) / px))))
    row_min = max(0, min(height - 1, int(math.floor((top - y_max) / px))))
    row_max = max(0, min(height - 1, int(math.floor((top - y_min) / px))))

    win_cols = col_max - col_min + 1
    win_rows = row_max - row_min + 1

    mask_flat = forest_mask.ravel(order="C")
    masked_positions = np.flatnonzero(mask_flat)
    idx_map = np.full(mask_flat.shape[0], -1, dtype=np.int64)
    idx_map[masked_positions] = np.arange(masked_positions.size, dtype=np.int64)

    rows = np.arange(row_min, row_max + 1, dtype=np.int64)
    cols = np.arange(col_min, col_max + 1, dtype=np.int64)
    rr, cc = np.meshgrid(rows, cols, indexing="ij")

    transform_window = rasterio.Affine(
        px, 0, left + col_min * px, 0, -px, top - row_min * px
    )
    poly_mask_window = rasterio.features.rasterize(
        [(polygon, 1)],
        out_shape=(win_rows, win_cols),
        transform=transform_window,
        fill=0,
        dtype="uint8",
    )

    inside_mask = poly_mask_window.astype(bool)
    full_flat_idx = (rr * width + cc).ravel()
    poly_flat_idx = full_flat_idx[inside_mask.ravel()]
    masked_idx_in_window = idx_map[poly_flat_idx]
    sel = masked_idx_in_window[masked_idx_in_window >= 0].tolist()

    if len(sel) == 0:
        raise RuntimeError("No masked pixels in polygon!")

    return sel

In [ ]:
anom = ds_temporal["anomalies"]
anom_scores = ds_temporal["anomaly_scores"]

In [ ]:
gdf = gpd.read_file("../data/event_polygons/polygons_drought.kml", driver="KML")
gdf = gdf.to_crs(epsg=2056)  # Swiss LV95
polygon = gdf.geometry.unary_union

# ----- polygon bounds -> pixel window -----
x_min, y_min, x_max, y_max = polygon.bounds

col_min = int(math.floor((x_min - REF_BBOX.left) / PX))
col_max = int(math.floor((x_max - REF_BBOX.left) / PX))
row_min = int(math.floor((REF_BBOX.top - y_max) / PX))
row_max = int(math.floor((REF_BBOX.top - y_min) / PX))

# clip to raster extent
col_min = max(0, min(width - 1, col_min))
col_max = max(0, min(width - 1, col_max))
row_min = max(0, min(height - 1, row_min))
row_max = max(0, min(height - 1, row_max))

win_cols = col_max - col_min + 1
win_rows = row_max - row_min + 1
print(
    f"Window cols {col_min}..{col_max} ({win_cols}), rows {row_min}..{row_max} ({win_rows})"
)

# ----- load mask -----
mask = forest_mask
assert mask.shape == (height, width), (
    f"Mask shape {mask.shape} != raster {(height, width)}"
)

mask_flat = mask.ravel(order="C")
masked_positions = np.flatnonzero(mask_flat)
n_masked = masked_positions.size
print(f"Mask has {n_masked} True pixels.")

# build index map from full array -> masked array
idx_map = np.full(mask_flat.shape[0], -1, dtype=np.int64)
idx_map[masked_positions] = np.arange(n_masked, dtype=np.int64)

# ----- compute pixel centers in window -----
rows = np.arange(row_min, row_max + 1, dtype=np.int64)
cols = np.arange(col_min, col_max + 1, dtype=np.int64)
rr, cc = np.meshgrid(rows, cols, indexing="ij")

xx = REF_BBOX.left + (cc + 0.5) * PX
yy = REF_BBOX.top - (rr + 0.5) * PX

poly_mask = rasterio.features.rasterize(
    [(polygon, 1)],
    out_shape=(height, width),
    transform=REF_TRANSFORM,
    fill=0,
    dtype="uint8",
)

# extract only window
inside_mask = poly_mask[row_min : row_max + 1, col_min : col_max + 1].astype(bool)

# flatten to keep same workflow
inside = inside_mask.ravel()

# ----- flat indices -----
full_flat_idx = (rr * width + cc).ravel()

# keep only pixels inside polygon
poly_flat_idx = full_flat_idx[inside]

masked_idx_in_window = idx_map[poly_flat_idx]
is_masked = masked_idx_in_window >= 0
n_masked_in_window = is_masked.sum()
print(
    f"Pixels in window: {full_flat_idx.size}, masked pixels in polygon: {n_masked_in_window}"
)

if n_masked_in_window == 0:
    raise RuntimeError("No masked pixels in polygon!")

sel_7 = masked_idx_in_window[is_masked].tolist()

drought_idx = np.array(sel_7, dtype=np.int64)
nodrought_idx = np.random.choice(anom.shape[0], size=drought_idx.size, replace=False)

anom_drought = anom[drought_idx, :]
anom_nodrought = anom[nodrought_idx, :]

anom_score_drought = anom_scores[drought_idx, :]
anom_score_nodrought = anom_scores[nodrought_idx, :]

print("Drought anomalies:", anom_drought.shape)
print("No-drought anomalies:", anom_nodrought.shape)

In [ ]:
# CLEARCUT

gdf = gpd.read_file("../data/event_polygons/polygons_clearcut.kml", driver="KML")
gdf = gdf.to_crs(epsg=2056)  # Swiss LV95
polygon = gdf.geometry.unary_union

sel_clearcut = get_pixels_from_polygon(
    polygon, forest_mask, REF_BBOX.left, REF_BBOX.top, PX, width, height
)

gdf = gpd.read_file("../data/event_polygons/polygons_noclearcut.kml", driver="KML")
gdf = gdf.to_crs(epsg=2056)  # Swiss LV95
polygon = gdf.geometry.unary_union

sel_noclearcut = get_pixels_from_polygon(
    polygon, forest_mask, REF_BBOX.left, REF_BBOX.top, PX, width, height
)

clearcut_idx = np.array(sel_clearcut, dtype=np.int64)
noclearcut_idx = np.array(sel_noclearcut, dtype=np.int64)

anom_clearcut = anom[clearcut_idx, :]
anom_noclearcut = anom[noclearcut_idx, :]

anom_score_clearcut = anom_scores[clearcut_idx, :]
anom_score_noclearcut = anom_scores[noclearcut_idx, :]

print("Cleancut anomalies:", anom_clearcut.shape)
print("No-clearcut anomalies:", anom_noclearcut.shape)

In [ ]:
# FIRE
gdf = gpd.read_file("../data/event_polygons/polygons_fire.kml", driver="KML")
gdf = gdf.to_crs(epsg=2056)  # Swiss LV95
polygon = gdf.geometry.unary_union

sel_fire = get_pixels_from_polygon(
    polygon, forest_mask, REF_BBOX.left, REF_BBOX.top, PX, width, height
)

gdf = gpd.read_file("../data/event_polygons/polygons_nofire.kml", driver="KML")
gdf = gdf.to_crs(epsg=2056)  # Swiss LV95
polygon = gdf.geometry.unary_union

sel_nofire = get_pixels_from_polygon(
    polygon, forest_mask, REF_BBOX.left, REF_BBOX.top, PX, width, height
)

print(f"Selected {len(sel_fire)} fire-affected pixels.")
print(f"Selected {len(sel_nofire)} no-fire-affected pixels.")

fire_idx = np.array(sel_fire, dtype=np.int64)
nofire_idx = np.array(sel_nofire, dtype=np.int64)
anom_fire = anom[fire_idx, :]
anom_nofire = anom[nofire_idx, :]
anom_score_fire = anom_scores[fire_idx, :]
anom_score_nofire = anom_scores[nofire_idx, :]
print("Fire anomalies:", anom_fire.shape)
print("No-fire anomalies:", anom_nofire.shape)

In [ ]:
# STORM

sel_storm = [
    int(line.strip()) for line in open("../data/event_polygons/pixels_storm.txt")
]

gdf = gpd.read_file("../data/event_polygons/polygons_nostorm.kml", driver="KML")
gdf = gdf.to_crs(epsg=2056)  # Swiss LV95
polygon = gdf.geometry.unary_union

sel_nostorm = get_pixels_from_polygon(
    polygon, forest_mask, REF_BBOX.left, REF_BBOX.top, PX, width, height
)

storm_idx = np.array(sel_storm, dtype=np.int64)
nostorm_idx = np.array(sel_nostorm, dtype=np.int64)
anom_storm = anom[storm_idx, :]
anom_nostorm = anom[nostorm_idx, :]
anom_score_storm = anom_scores[storm_idx, :]
anom_score_nostorm = anom_scores[nostorm_idx, :]
print("Storm anomalies:", anom_storm.shape)
print("No-storm anomalies:", anom_nostorm.shape)

In [ ]:
# BEETLES

gdf = gpd.read_file("../data/event_polygons/polygons_beetles.kml", driver="KML")
gdf = gdf.to_crs(epsg=2056)  # Swiss LV95
polygon = gdf.geometry.unary_union

sel_beetles = get_pixels_from_polygon(
    polygon, forest_mask, REF_BBOX.left, REF_BBOX.top, PX, width, height
)

gdf = gpd.read_file("../data/event_polygons/polygons_nobeetles.kml", driver="KML")
gdf = gdf.to_crs(epsg=2056)  # Swiss LV95
polygon = gdf.geometry.unary_union

sel_nobeetles = get_pixels_from_polygon(
    polygon, forest_mask, REF_BBOX.left, REF_BBOX.top, PX, width, height
)

beetles_idx = np.array(sel_beetles, dtype=np.int64)
nobeetles_idx = np.array(sel_nobeetles, dtype=np.int64)
anom_beetles = anom[beetles_idx, :]
anom_nobeetles = anom[nobeetles_idx, :]
anom_score_beetles = anom_scores[beetles_idx, :]
anom_score_nobeetles = anom_scores[nobeetles_idx, :]
print("Beetles anomalies:", anom_beetles.shape)
print("No-beetles anomalies:", anom_nobeetles.shape)

In [ ]:
dates_sorted_idx = np.argsort(dates)
dates_sorted = np.array(dates)[dates_sorted_idx]
dates_sorted = pd.to_datetime(dates_sorted)

anom_fire_sorted = anom_fire[:, dates_sorted_idx]
anom_nofire_sorted = anom_nofire[:, dates_sorted_idx]
anom_drought_sorted = anom_drought[:, dates_sorted_idx]
anom_nodrought_sorted = anom_nodrought[:, dates_sorted_idx]
anom_storm_sorted = anom_storm[:, dates_sorted_idx]
anom_nostorm_sorted = anom_nostorm[:, dates_sorted_idx]
anom_clearcut_sorted = anom_clearcut[:, dates_sorted_idx]
anom_noclearcut_sorted = anom_noclearcut[:, dates_sorted_idx]
anom_beetles_sorted = anom_beetles[:, dates_sorted_idx]
anom_nobeetles_sorted = anom_nobeetles[:, dates_sorted_idx]

anom_score_fire_sorted = anom_score_fire[:, dates_sorted_idx]
anom_score_nofire_sorted = anom_score_nofire[:, dates_sorted_idx]
anom_score_drought_sorted = anom_score_drought[:, dates_sorted_idx]
anom_score_nodrought_sorted = anom_score_nodrought[:, dates_sorted_idx]
anom_score_storm_sorted = anom_score_storm[:, dates_sorted_idx]
anom_score_nostorm_sorted = anom_score_nostorm[:, dates_sorted_idx]
anom_score_clearcut_sorted = anom_score_clearcut[:, dates_sorted_idx]
anom_score_noclearcut_sorted = anom_score_noclearcut[:, dates_sorted_idx]
anom_score_beetles_sorted = anom_score_beetles[:, dates_sorted_idx]
anom_score_nobeetles_sorted = anom_score_nobeetles[:, dates_sorted_idx]

In [ ]:
def get_dates_and_fraction_neg_anomalies(anom_subset, dates):
    """Get the fraction of negative anomalies per timestep, ignoring invalid values, and return the dates and fractions for valid timesteps.

    Args:
        anom_subset (numpy.ndarray): A 2D array of shape (n_pixels, n_time) containing anomaly values, where -1 indicates a negative anomaly, and -128/127 indicate invalid data.
        dates (numpy.ndarray): A 1D array of shape (n_time,) containing the corresponding dates for each timestep.

    Returns:
        tuple: A tuple containing the dates and fractions for valid timesteps.
    """
    valid_mask = (anom_subset != -128) & (anom_subset != 127)
    neg_mask = (anom_subset == -1) & valid_mask

    # Fraction of valid pixels showing negative anomaly per timestep
    frac = np.sum(neg_mask, axis=0) / np.sum(valid_mask, axis=0)

    plot_mask = np.where(~np.isnan(frac), True, False)

    return dates[plot_mask], frac[plot_mask]


def get_score_summary(anom_subset, dates):
    """Get the median score and percentiles per timestep, ignoring invalid values.

    Args:
        anom_subset (numpy.ndarray): A 2D array of shape (n_pixels, n_time) containing anomaly values, where -1 indicates a negative anomaly, and -128/127 indicate invalid data.
        dates (numpy.ndarray): A 1D array of shape (n_time,) containing the corresponding dates for each timestep.

    Returns:
        tuple: A tuple containing the dates, median scores, and percentiles for valid timesteps.
    """
    # Compute statistics per timestep
    median_score = np.nanmedian(anom_subset, axis=0)
    p25 = np.nanpercentile(anom_subset, 25, axis=0)
    p75 = np.nanpercentile(anom_subset, 75, axis=0)

    # Mask where all values are nan
    plot_mask = ~np.isnan(median_score)

    return dates[plot_mask], median_score[plot_mask], p25[plot_mask], p75[plot_mask]

In [ ]:
plt.rcParams.update(
    {
        "font.size": 12,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    }
)


def format_date_axis(ax):
    """Format the x-axis of a plot to show years with appropriate ticks and gridlines.

    Args:
        ax (matplotlib.axes.Axes): The axes object to format.
    """
    ax.xaxis.set_major_locator(matplotlib.dates.YearLocator())
    ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%Y"))
    ax.grid(which="major", axis="x", linestyle="--", alpha=0.6)
    for label in ax.get_xticklabels():
        label.set_ha("left")


fig, axs = plt.subplots(
    2,
    5,
    figsize=(20, 7.5),
    sharey="row",
    sharex="col",
    dpi=600,
)

plt.subplots_adjust(wspace=0.15, hspace=0.15)

pal = {
    "control": "#333333",
    "Fire": "#DD8452",
    "Drought": "#C44E52",
    "Storm": "#8172B2",
    "Clearcut": "#55A868",
    "Beetles": "#937860",
}

cases = [
    (
        "Fire",
        "2022-01-01",
        "2025-01-01",
        anom_score_nofire_sorted,
        anom_score_fire_sorted,
        anom_nofire_sorted,
        anom_fire_sorted,
        [("2023-07-17",)],
    ),
    (
        "Drought",
        "2017-01-01",
        "2020-01-01",
        anom_score_nodrought_sorted,
        anom_score_drought_sorted,
        anom_nodrought_sorted,
        anom_drought_sorted,
        [("2018-07-29",), ("2018-09-09",)],
    ),
    (
        "Storm",
        "2017-01-01",
        "2020-01-01",
        anom_score_nostorm_sorted,
        anom_score_storm_sorted,
        anom_nostorm_sorted,
        anom_storm_sorted,
        [("2018-01-02",)],
    ),
    (
        "Clearcut",
        "2020-01-01",
        "2023-01-01",
        anom_score_noclearcut_sorted,
        anom_score_clearcut_sorted,
        anom_noclearcut_sorted,
        anom_clearcut_sorted,
        [("2021-01-01",)],
    ),
    (
        "Beetles",
        "2020-01-01",
        "2023-01-01",
        anom_score_nobeetles_sorted,
        anom_score_beetles_sorted,
        anom_nobeetles_sorted,
        anom_beetles_sorted,
        [("2021-01-01",)],
    ),
]

for col, (
    label,
    start,
    end,
    score_ctrl,
    score_case,
    frac_ctrl,
    frac_case,
    vlines,
) in enumerate(cases):
    mask = (dates_sorted >= pd.Timestamp(start)) & (dates_sorted < pd.Timestamp(end))

    ax = axs[0, col]
    d, m, p25, p75 = get_score_summary(score_ctrl[:, mask], dates_sorted[mask])
    ax.plot(d, m, color=pal["control"], lw=2.2, ls="--")
    ax.fill_between(d, p25, p75, color=pal["control"], alpha=0.18)
    d, m, p25, p75 = get_score_summary(score_case[:, mask], dates_sorted[mask])
    ax.plot(d, m, color=pal[label], lw=2.2)
    ax.fill_between(d, p25, p75, color=pal[label], alpha=0.18)

    for (v,) in vlines:
        ax.axvline(pd.Timestamp(v), color="black", ls="--", lw=1.2)

    ax.set_title(label, fontsize=14)
    if col == 0:
        ax.set_ylabel("NDVI anomaly score", fontsize=12)

    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.22), ncol=2, frameon=False)
    format_date_axis(ax)

    ax = axs[1, col]
    d, f = get_dates_and_fraction_neg_anomalies(frac_ctrl[:, mask], dates_sorted[mask])
    ax.plot(d, f, color=pal["control"], lw=2.2, ls="--")
    d, f = get_dates_and_fraction_neg_anomalies(frac_case[:, mask], dates_sorted[mask])
    ax.plot(d, f, color=pal[label], lw=2.2)

    for (v,) in vlines:
        ax.axvline(pd.Timestamp(v), color="black", ls="--", lw=1.2)

    if col == 0:
        ax.set_ylabel("Fraction of pixels\nwith negative anomaly", fontsize=14)

    # set tick label sizes
    ax.tick_params(axis="x", labelsize=12)
    ax.tick_params(axis="y", labelsize=12)

    ax.legend(loc="upper left", frameon=False)
    format_date_axis(ax)

fig.suptitle("NDVI anomaly case studies", fontsize=16)
fig.savefig("../figs/figure_6.png", dpi=300)

# Figure 7

In [ ]:
habitat_dict = {
    1: "Water",
    2: "Riparian & Wetland",
    3: "Glacier & Rock",
    4: "Grassland",
    5: "Shrubland",
    6: "Forest",
    7: "Pioneer Vegetation",
    8: "Agricultural Area",
    9: "Urban Area",
    60: "Forest Plantation",
    61: "March & Floodplain Forest",
    62: "Beech Forest",
    63: "Other Deciduous Forest",
    64: "Warm-Loving Pine Forest",
    65: "Raised Bog Forest",
    66: "Mountain Coniferous Forest",
    612: "Softwood Floodplain Forest",
    614: "Hardwood Floodplain Forest",
    621: "Orchid Beech Forest",
    622: "Woodrush Beech Forest",
    623: "Woodruff Beech Forest",
    624: "Toothwort Beech Forest",
    625: "Fir-Beech Forest",
    631: "Maple Ravine Forest",
    632: "Linden Mixed Forest",
    633: "Oak-Hornbeam Forest",
    634: "Downy Oak Forest",
    635: "Hop Hornbeam Forest",
    636: "Acidic Oak Mixed Forest",
    637: "Chestnut Forest",
    638: "Deciduous Forest & Evergreen Shrubs",
    639: "Robinia Forest",
    641: "Pipe Grass Pine Forest",
    642: "Subcontintental Calcareous Pine Forest",
    643: "Continental Steppe Pine Forest",
    651: "Raised Bog Birch Forest",
    652: "Raised Moor Mountain Pine Forest",
    653: "Raised Moor Spruce Forest",
    661: "Fir-Spruce Forest",
    662: "Blueberry-Spruce Forest",
    663: "Larch-Swiss Pine Forest",
    664: "Larch Forest",
    665: "Mountain Pine Forest",
    60000: "Single-tree, non-complex",
    60001: "Single-tree, complex",
    65535: "No Data / Unclassified",
}
habitat_codes = list(habitat_dict.keys())


def load_habitat_embedding_matrix(checkpoint_path, expected_n=None):
    """Load the habitat embedding matrix from a checkpoint file."""
    sd = torch.load(checkpoint_path, map_location="cpu")
    keys = [k for k in sd.keys() if "habitat" in k and "weight" in k]
    if len(keys) == 1:
        w = sd[keys[0]]
    else:
        cands = []
        for k in keys:
            t = sd[k]
            if t.ndim == 2:
                if expected_n is None or t.shape[0] == expected_n:
                    cands.append((k, t))
        w = cands[0][1]
    return w.detach().cpu().numpy()


def run_tsne(emb, perplexity, random_state):
    """Run t-SNE on the given embedding matrix with specified perplexity and random state."""
    tsne = sklearn.manifold.TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=random_state,
    )
    return tsne.fit_transform(emb)


def forest_category(code):
    """Determine the forest category for a given habitat code."""
    deciduous = {63, 631, 632, 633, 634, 635, 636, 637, 638, 639}
    beech = {62, 621, 622, 623, 624, 625}
    hl_pine = {64, 641, 642, 643}
    conifer = {66, 661, 662, 663, 664, 665}
    raised_bog = {65, 651, 652, 653}
    march_floodplain = {61, 612, 614}
    special = {60, 60000, 60001}
    if code in beech:
        return "Beech forest"
    elif code in deciduous:
        return "Other deciduous forest"
    elif code in conifer:
        return "Mountain coniferous forest"
    elif code in raised_bog:
        return "Raised bog forest"
    elif code in hl_pine:
        return "Warm-loving pine forest"
    elif code in march_floodplain:
        return "March / floodplain forest"
    elif code in special:
        return "Forest plantation / single-tree"
    else:
        return "Other"


FOREST_COLORS = {
    "Beech forest": "#1f77b4",
    "Other deciduous forest": "#ff7f0e",
    "Mountain coniferous forest": "#2ca02c",
    "Raised bog forest": "#d62728",
    "Warm-loving pine forest": "#9467bd",
    "March / floodplain forest": "#8c564b",
    "Mixed forest": "#e377c2",
    "Forest plantation / single-tree": "#7f7f7f",
    "Other": "#bcbd22",
}

In [ ]:
emb = load_habitat_embedding_matrix(
    "../checkpoints/encoder.pt", expected_n=len(habitat_codes)
)
coords = run_tsne(emb, perplexity=5, random_state=42)

labels, categories, codes, xs, ys = [], [], [], [], []
for code, (x, y) in zip(habitat_codes, coords):
    cat = forest_category(code)
    if cat is not None:
        labels.append(f"{habitat_dict[code]} ({code})")
        categories.append(cat)
        codes.append(code)
        xs.append(x)
        ys.append(y)

df = pd.DataFrame(
    {"code": codes, "name": labels, "category": categories, "x": xs, "y": ys}
)
points = df[["x", "y"]].to_numpy()

fig, ax = plt.subplots(figsize=(12, 10))

for cat in np.unique(categories):
    idx = [i for i, c in enumerate(categories) if c == cat]
    ax.scatter(
        points[idx, 0],
        points[idx, 1],
        label=cat,
        s=100,
        color=FOREST_COLORS.get(cat, "gray"),
        alpha=0.9,
        edgecolor="black",
        linewidths=0.6,
    )

texts = []
for i, lab in enumerate(labels):
    txt = ax.text(
        points[i, 0],
        points[i, 1],
        lab,
        fontsize=10,
        fontweight="semibold",
        color="black",
        ha="center",
        va="center",
    )
    txt.set_path_effects([pe.withStroke(linewidth=2, foreground="white")])
    texts.append(txt)

adjustText.adjust_text(
    texts,
    x=points[:, 0],
    y=points[:, 1],
    arrowprops=dict(
        arrowstyle="-|>", color="gray", lw=0.7, alpha=0.7, connectionstyle="arc3,rad=0"
    ),
    expand_points=(1.5, 1.7),
    expand_text=(1.2, 1.4),
    force_text=0.5,
    force_points=0.3,
    lim=3000,
)

ax.legend(frameon=False, fontsize=11)
ax.set_xticks([])
ax.set_yticks([])
fig.tight_layout()
fig.savefig("../figs/figure_7.png", dpi=300)
plt.show()